# Any2Reg Demo

Minimal inference for cardiac MRI groupwise registration: one STONE case and one ACDC case. Outputs PNG + GIF per method (raw baseline + Any2RegNet). Metrics are sanity checks only.

In [ ]:
import sys
from pathlib import Path
import json
from datetime import datetime

package_root = Path.cwd().parent
if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

import torch
from any2reg_submission import io, infer, visualize, eval_simple

# Paths: prefer sample_data in this repo
sample_stone = package_root / "sample_data" / "stone"
sample_acdc = package_root / "sample_data" / "acdc"
use_sample = (sample_stone / "data").exists() and (sample_acdc / "data").exists()
stone_root = sample_stone if use_sample else Path("/path/to/stone")
acdc_root = sample_acdc if use_sample else Path("/path/to/acdc")
stone_feature_dir = stone_root / "features" if (stone_root / "features").exists() else None
acdc_feature_dir = acdc_root / "feature" if (acdc_root / "feature").exists() else None
ckpt_path = package_root / "checkpoints" / "any2regnet_raw_logits_best.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = package_root / "outputs" / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
output_dir.mkdir(parents=True, exist_ok=True)

config = {"stone_root": stone_root, "acdc_root": acdc_root, "stone_feature_dir": stone_feature_dir,
          "acdc_feature_dir": acdc_feature_dir, "any2regnet_checkpoint": ckpt_path, "output_dir": output_dir, "device": device}
with open(output_dir / "config.json", "w") as f:
    json.dump({k: str(v) for k, v in config.items()}, f, indent=2)
print("Config:", {k: str(v) for k, v in config.items()})

## 1. Resolve cases & load data

In [ ]:
# Resolve one STONE case (138_4) and one ACDC case (first subject, one slice)
cases = {}
stone_case = io.resolve_stone_case(config["stone_root"], subject_token="138_4")
acdc_case = io.resolve_acdc_case(config["acdc_root"], select_mid_slice=True)
if stone_case:
    cases["stone"] = stone_case
if acdc_case:
    cases["acdc"] = acdc_case
if not cases:
    raise RuntimeError("No cases found. Check data paths.")

# Load images and optional features
loaded_data = {}
for name, path in cases.items():
    data = io.load_nifti_case(path)
    feat_dir = config.get(f"{name}_feature_dir")
    base = path.stem.replace(".nii", "").replace("_0000", "")
    feat_path = Path(feat_dir) / f"{base}_features.npz" if feat_dir else None
    data["features"] = io.load_feature_map(feat_path, feature_key="logits_final") if feat_path and feat_path.exists() else None
    loaded_data[name] = data
    print(f"{name}: {path.name} shape {data['images'].shape}")

with open(config["output_dir"] / "cases.json", "w") as f:
    json.dump({n: {"path": str(p), "filename": p.name} for n, p in cases.items()}, f, indent=2)

## 2. Inference (raw baseline + Any2RegNet)

In [ ]:
inference_results = {}
for name, data in loaded_data.items():
    inference_results[name] = {
        "raw": infer.run_raw_baseline(data["images"], device=config["device"]),
        "any2regnet": infer.run_any2regnet_inference(
            data["images"], feature_maps=data["features"],
            checkpoint_path=config["any2regnet_checkpoint"], device=config["device"], num_iters=1,
        ),
    }
    print(f"{name}: raw + any2regnet done")

## 3. Evaluate & visualize

In [ ]:
# Sanity-check metrics (not for rigorous comparison)
all_metrics = []
for name, results_dict in inference_results.items():
    for method, result in results_dict.items():
        m = eval_simple.evaluate_case(result, masks_original=None, method_name=f"{name}_{method}")
        all_metrics.append(m)
print(eval_simple.format_results_table(all_metrics))
with open(config["output_dir"] / "metrics.txt", "w") as f:
    f.write(eval_simple.format_results_table(all_metrics))
with open(config["output_dir"] / "metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

# PNG + GIF per dataset and method
vis_dir = config["output_dir"] / "visualizations"
vis_dir.mkdir(exist_ok=True)
for name, results_dict in inference_results.items():
    data = loaded_data[name]
    for method, result in results_dict.items():
        visualize.visualize_group_result(
            data["images"], result["warped"], result["disp"], result["template"],
            vis_dir / name, name=method, grid_step=16,
        )
print("Outputs:", config["output_dir"])

**Done.** Outputs: `config.json`, `cases.json`, `metrics.json`/`.txt`, `visualizations/<dataset>/{raw,any2regnet}.png` + `.gif`.